# TextSplitter のソースコード解析

内部の3つの主要メソッドについて説明します：

メソッド1：
split_text(self, text: str) -> list[str]:
> 渡す引数の型：テキスト内容（または文字列）、戻り値の型：文字列のリスト
>
> このメソッドは抽象メソッドで、具体的な実装の詳細はサブクラスが決定する

メソッド2：
create_documents(self, texts: list[str],...) -> list[Document]:
> 渡す引数の型：文字列のリスト、戻り値の型：Document オブジェクトのリスト
>
> このメソッドは内部で split_text() を呼び出す。つまり引数の各文字列を split_text() に渡して実行し、得られた文字列のリストの各文字列を Document オブジェクトにラップすることで list[Document] を構成する。


メソッド3：
split_documents(self, documents: Iterable[Document]) -> list[Document]:
> 渡す引数の型：Document オブジェクトのリスト、戻り値の型：Document オブジェクトのリスト
>
> このメソッドは内部で create_documents() を呼び出す。引数の各 Document オブジェクトから page_content フィールドを抽出して文字列のリストを構成し、その後メソッド2を呼び出す。

# 具体的なドキュメント分割器の使用


## 1、CharacterTextSplitter：Split by character

例1：文字列テキストの分割

In [1]:
# 1.関連する依存関係をインポート
from langchain_text_splitters import CharacterTextSplitter

# 2.サンプルテキスト
text = """
LangChain は、言語モデルによって駆動されるアプリケーションを開発するためのフレームワークです。ツールと抽象化の一式を提供し、開発者がより簡単に複雑なアプリケーションを構築できるようにします。
"""

# 3.文字分割器を定義
splitter = CharacterTextSplitter(
    chunk_size=50, # 各チャンクのサイズ
    chunk_overlap=5,# チャンク間の重複文字数
    # length_function=len,
    separator=""   # 空文字列に設定すると、区切り文字優先を無効にすることを意味する
)

# 4.テキストを分割
texts = splitter.split_text(text)

# 5.結果を出力
for i, chunk in enumerate(texts):
    print(f"チャンク {i+1}:長さ：{len(chunk)}")
    print(chunk)
    print("-" * 50)

チャンク 1:長さ：49
LangChain は、言語モデルによって駆動されるアプリケーションを開発するためのフレームワーク
--------------------------------------------------
チャンク 2:長さ：50
ームワークです。ツールと抽象化の一式を提供し、開発者がより簡単に複雑なアプリケーションを構築できるよ
--------------------------------------------------
チャンク 3:長さ：11
築できるようにします。
--------------------------------------------------


例2：区切り文字を指定

In [2]:
# 1.関連する依存関係をインポート
from langchain_text_splitters import CharacterTextSplitter

# 2.分割するテキストを定義
text = "これはサンプルテキストです。CharacterTextSplitter を使ってこれを小さなチャンクに分割します。分割は文字数に基づきます。"

# text = """
# LangChain は、言語モデルによって駆動されるアプリケーションを開発するためのフレームワークです。ツールと抽象化の一式を提供します。開発者がより簡単に複雑なアプリケーションを構築できるようにします。
# """

# 3.分割器のインスタンスを定義
text_splitter = CharacterTextSplitter(
    chunk_size=30,   # 各チャンクの最大文字数
    chunk_overlap=5, # チャンク間の重複文字数
    separator="。",  # 句点での分割を優先
)

# 4.分割を開始
chunks = text_splitter.split_text(text)

# 5.結果を出力
for  i,chunk in enumerate(chunks):
    print(f"チャンク {i + 1}:長さ：{len(chunk)}")
    print(chunk)
    print("-"*50)


Created a chunk of size 42, which is longer than the specified 30


チャンク 1:長さ：13
これはサンプルテキストです
--------------------------------------------------
チャンク 2:長さ：42
CharacterTextSplitter を使ってこれを小さなチャンクに分割します
--------------------------------------------------
チャンク 3:長さ：12
分割は文字数に基づきます
--------------------------------------------------


**separator 優先の原則**：`separator`（例："。"）が設定されている場合、分割器はまず区切り文字の位置で分割を試み、その後 chunk_size を考慮します。これは文の途中で無理に切断することを避けるためです。この設計は以下を目的としています：

1. 意味的な完全性を優先的に保つ（文を途中で切らない）
2. 意味のない断片（単語の半分/不完全な文など）が生じるのを避ける
3. `chunk_size` が断片より小さい場合、断片を分割できず、overlap が無効になる。
4. chunk_overlap は結合後の断片間でのみ有効（`chunk_size` が十分大きい場合）。結合された断片がない場合、overlap は無効になる。

例3：区切り文字を指定

In [3]:
# 1.関連する依存関係をインポート
from langchain_text_splitters import CharacterTextSplitter

# 2.分割するテキストを定義
text = "これは1段落目のテキストです。これは2段落目の内容です。最後の段落で終わります。"

# 3.文字分割器を定義
text_splitter = CharacterTextSplitter(
    chunk_size=20,
    chunk_overlap=8,
    separator="。",
    # keep_separator=True #チャンク内に区切り文字を保持するかどうか
)

# 4.テキストを分割
chunks = text_splitter.split_text(text)

# 5.結果を出力
for  i,chunk in enumerate(chunks):
    print(f"チャンク {i + 1}:長さ：{len(chunk)}")
    print(chunk)
    print("-"*50)

チャンク 1:長さ：14
これは1段落目のテキストです
--------------------------------------------------
チャンク 2:長さ：12
これは2段落目の内容です
--------------------------------------------------
チャンク 3:長さ：11
最後の段落で終わります
--------------------------------------------------


## 2、RecursiveCharacterTextSplitter：最もよく使われる

例1：split_text() メソッドを使ったデモ

In [4]:
# 1.関連する依存関係をインポート
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.RecursiveCharacterTextSplitter 分割器オブジェクトを定義
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    add_start_index=True,
)

# 3.分割する内容を定義
text="LangChain フレームワークの特徴\n\nマルチモデル統合(GPT/Claude)\n記憶管理機能\nチェーン呼び出し設計。文書分析シナリオの例：PDF/Word などの形式を処理する必要がある。"

# 4.分割器で分割
paragraphs = text_splitter.split_text(text)

for i,chunk in enumerate(paragraphs):
    print(f"チャンク{i + 1}、長さ：{len(chunk)}")
    print(chunk)
    print('-' * 50)

チャンク1、長さ：9
LangChain
--------------------------------------------------
チャンク2、長さ：9
フレームワークの特
--------------------------------------------------
チャンク3、長さ：1
徴
--------------------------------------------------
チャンク4、長さ：9
マルチモデル統合(
--------------------------------------------------
チャンク5、長さ：10
GPT/Claude
--------------------------------------------------
チャンク6、長さ：1
)
--------------------------------------------------
チャンク7、長さ：6
記憶管理機能
--------------------------------------------------
チャンク8、長さ：9
チェーン呼び出し設
--------------------------------------------------
チャンク9、長さ：10
計。文書分析シナリオ
--------------------------------------------------
チャンク10、長さ：10
の例：PDF/Wor
--------------------------------------------------
チャンク11、長さ：1
d
--------------------------------------------------
チャンク12、長さ：9
などの形式を処理す
--------------------------------------------------
チャンク13、長さ：7
る必要がある。
--------------------------------------------------


例2：create_documents() メソッドを使ったデモ。文字列のリストを渡し、Document オブジェクトのリストを返す

In [5]:
# 1.関連する依存関係をインポート
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.RecursiveCharacterTextSplitter 分割器オブジェクトを定義
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    add_start_index=True,
)

# 3.分割する内容を定義
# text="LangChain フレームワークの特徴\n\nマルチモデル統合(GPT/Claude)\n記憶管理機能\nチェーン呼び出し設計。文書分析シナリオの例：PDF/Word などの形式を処理する必要がある。"

list=["LangChain フレームワークの特徴\n\nマルチモデル統合(GPT/Claude)\n記憶管理機能\nチェーン呼び出し設計。文書分析シナリオの例：PDF/Word などの形式を処理する必要がある。"]

# 4.分割器で分割
# create_documents()：仮引数は文字列のリスト、戻り値は Document のリスト
paragraphs = text_splitter.create_documents(list)


for para in paragraphs:
    print(para)
    print('-------')

page_content='LangChain' metadata={'start_index': 0}
-------
page_content='フレームワークの特' metadata={'start_index': 10}
-------
page_content='徴' metadata={'start_index': 19}
-------
page_content='マルチモデル統合(' metadata={'start_index': 22}
-------
page_content='GPT/Claude' metadata={'start_index': 31}
-------
page_content=')' metadata={'start_index': 41}
-------
page_content='記憶管理機能' metadata={'start_index': 43}
-------
page_content='チェーン呼び出し設' metadata={'start_index': 50}
-------
page_content='計。文書分析シナリオ' metadata={'start_index': 59}
-------
page_content='の例：PDF/Wor' metadata={'start_index': 69}
-------
page_content='d' metadata={'start_index': 79}
-------
page_content='などの形式を処理す' metadata={'start_index': 81}
-------
page_content='る必要がある。' metadata={'start_index': 90}
-------


例3：create_documents() メソッドを使ったデモ。ローカルファイルの内容を文字列として読み込み、分割する

In [7]:
# 1.関連する依存関係をインポート
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2..txt ファイルを開く
with open("../asset/load/09-ai.txt", encoding="utf-8") as f:
    state_of_the_union = f.read()  #返り値は文字列

# 3.RecursiveCharacterTextSplitter（再帰的文字分割器）を定義
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    #chunk_overlap=0,
    length_function=len
)

# 4.テキストを分割
texts = text_splitter.create_documents([state_of_the_union])

# 5.分割したテキストを出力
for text in texts:
    print(f"🔥{text.page_content}")

🔥人工智能（AI）是什么？
🔥人工智能（Artificial
🔥Intelligence，简称AI）是指由计算机系统模拟人类智能的技术，使其能够执行通常需要人类认知能力的任务，如学习、推理、决策和语言理解。AI的核心目标是让机器具备感知环境、处理信息并自主行动的
🔥让机器具备感知环境、处理信息并自主行动的能力。
🔥1. AI的技术基础
AI依赖多种关键技术：

机器学习（ML）：通过算法让计算机从数据中学习规律，无需显式编程。例如，推荐系统通过用户历史行为预测偏好。
🔥深度学习：基于神经网络的机器学习分支，擅长处理图像、语音等复杂数据。AlphaGo击败围棋冠军便是典型案例。

自然语言处理（NLP）：使计算机理解、生成人类语言，如ChatGPT的对话能力。
🔥2. AI的应用场景
AI已渗透到日常生活和各行各业：

医疗：辅助诊断（如AI分析医学影像）、药物研发加速。

交通：自动驾驶汽车通过传感器和AI算法实现安全导航。
🔥金融：欺诈检测、智能投顾（如风险评估模型）。

教育：个性化学习平台根据学生表现调整教学内容。

3. AI的挑战与未来
尽管前景广阔，AI仍面临问题：
🔥伦理争议：数据隐私、算法偏见（如招聘AI歧视特定群体）。

就业影响：自动化可能取代部分人工岗位，但也会创造新职业。

技术瓶颈：通用人工智能（AGI）尚未实现，当前AI仅擅长特定任务。
🔥未来，AI将与人类协作而非替代：医生借助AI提高诊断效率，教师利用AI定制课程。其发展需平衡技术创新与社会责任，确保技术造福全人类。


例4：split_documents() メソッドを使ったデモ。PDFLoader で文書を読み込み、その内容を再帰的分割器で分割する

In [ ]:
# 1.関連する依存関係をインポート
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.PyPDFLoader ローダーを定義
loader = PyPDFLoader("../asset/load/04-load.pdf")

# 3.文書オブジェクトを読み込み・分割
docs = loader.load()   # Document オブジェクトからなる list を返す
# print(f"1ページ目：\n{docs[0]}")

# 4.分割器を定義
text_splitter = RecursiveCharacterTextSplitter(
    # chunk_size=200,
    chunk_size=120,
    chunk_overlap=0,
    # chunk_overlap=100,
    length_function=len,
    add_start_index=True,
)

# 5.PDF の内容を分割して文書オブジェクトを取得
paragraphs = text_splitter.split_documents(docs)

for para in paragraphs:
    print(para)
    print('-------')

## 3、TokenTextSplitter/CharacterTextSplitter：Split by tokens

例1：TokenTextSplitter を使用

In [8]:
# 1.関連する依存関係をインポート
from langchain_text_splitters import TokenTextSplitter

# 2.TokenTextSplitter を初期化
text_splitter = TokenTextSplitter(
    chunk_size=33,  # 最大トークン数は 33
    chunk_overlap=0, # 重複トークン数は 0
    # model_name="gpt-4", # GPT-4 モデルのエンコーダーを選択
    encoding_name="cl100k_base",  # OpenAI のエンコーダーを使用し、テキストをトークン列に変換
)
# 3.テキストを定義
text = "人工知能は強力な開発フレームワークです。多様な言語モデルとツールチェーンをサポートします。人工知能とは、コンピュータプログラムによって人間の知能をシミュレートする科学のことです。1950年代の誕生以来、人工知能は幾度もの浮き沈みを経験してきました。"

# 4.分割を開始
texts = text_splitter.split_text(text)

# 分割結果を出力
print(f"元のテキストは {len(texts)} 個のチャンクに分割されました:")
for i, chunk in enumerate(texts):
    print(f"チャンク {i+1}: 長さ：{len(chunk)} 内容：{chunk}")
    print("-" * 50)

元のテキストは 4 個のチャンクに分割されました:
チャンク 1: 長さ：27 内容：人工知能は強力な開発フレームワークです。多様な言語モデ
--------------------------------------------------
チャンク 2: 長さ：37 内容：ルとツールチェーンをサポートします。人工知能とは、コンピュータプログラムに
--------------------------------------------------
チャンク 3: 長さ：37 内容：よって人間の知能をシミュレートする科学のことです。1950年代の誕生以来、
--------------------------------------------------
チャンク 4: 長さ：23 内容：人工知能は幾度もの浮き沈みを経験してきました。
--------------------------------------------------


例2：CharacterTextSplitter を使用

In [9]:
# 1.関連する依存関係をインポート
from langchain_text_splitters import CharacterTextSplitter
import tiktoken  # トークン数を計算するために使用


# 2.Token ベースの分割器を定義
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", # OpenAI のエンコーダーを使用
    chunk_size=18,
    chunk_overlap=0,
    separator="。",  # 句点を区切り文字として指定
    keep_separator=False,  # チャンク内に区切り文字を保持するかどうか
)
# 3.テキストを定義
text = "人工知能は強力な開発フレームワークです。多様な言語モデルとツールチェーンをサポートします。今日は天気がいいので、ピクニックに出かけたいです。でも面倒でやっぱり出かけたくない、どうしよう"

# 4.分割を開始
texts = text_splitter.split_text(text)

print(f"元のテキストは {len(texts)} 個のチャンクに分割されました:")
for i, chunk in enumerate(texts):
    print(f"チャンク {i+1}: 長さ：{len(chunk)} 内容：{chunk}")
    print("-" * 50)

Created a chunk of size 21, which is longer than the specified 18
Created a chunk of size 25, which is longer than the specified 18
Created a chunk of size 22, which is longer than the specified 18


元のテキストは 4 個のチャンクに分割されました:
チャンク 1: 長さ：19 内容：人工知能は強力な開発フレームワークです
--------------------------------------------------
チャンク 2: 長さ：24 内容：多様な言語モデルとツールチェーンをサポートします
--------------------------------------------------
チャンク 3: 長さ：24 内容：今日は天気がいいので、ピクニックに出かけたいです
--------------------------------------------------
チャンク 4: 長さ：22 内容：でも面倒でやっぱり出かけたくない、どうしよう
--------------------------------------------------


## 4、SemanticChunker：意味に基づくチャンク分割

例：

In [10]:
# pip install langchain_experimental
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain.embeddings import init_embeddings
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# テキストを読み込む
with open("../asset/load/09-ai1.txt", encoding="utf-8") as f:
    state_of_the_union = f.read()  #文字列を返す

# 埋め込みモデルを取得
embedding_model = init_embeddings(
	model="text-embedding-3-large",
    provider="openai",
	api_key=os.getenv("OPENROUTER_API_KEY"),
	base_url=os.getenv("OPENROUTER_API_BASE"),
)

# embedding_model = OpenAIEmbeddings(
#     model="BAAI/bge-m3", # 有料モデル ID： Pro/BAAI/bge-m3
#     base_url=os.getenv("SILICONFLOW_BASE_URL"),
#     api_key=os.getenv("SILICONFLOW_API_KEY"),
#     dimensions=1024
# )


# 分割器を取得
text_splitter = SemanticChunker(
    embeddings=embedding_model,
    breakpoint_threshold_type="percentile", # 分割点の閾値タイプ：["パーセンタイル", "標準偏差", "四分位範囲", "勾配"] のいずれかを選択
    breakpoint_threshold_amount=65.0, # 分割点の閾値数値（極めて低い閾値 → 高い分割感度）
    sentence_split_regex=r"(?<=[。？！])\s+" # 文分割の正規表現：句点・感嘆符・疑問符（。？！）の後にスペースがある場合、まずそれを独立した“文”として分割する。
)

# 文書を分割
docs = text_splitter.create_documents(texts = [state_of_the_union])

print(len(docs))
for doc in docs:
    print(f"🔍 文書: {doc}")

7
🔍 文書: page_content='人工智能综述：发展、应用与未来展望

摘要
人工智能（Artificial Intelligence，AI）作为计算机科学的一个重要分支，近年来取得了突飞猛进的发展。本文综述了人工智能的发展历程、核心技术、应用领域以及未来发展趋势。通过对人工智能的定义、历史背景、主要技术（如机器学习、深度学习、自然语言处理等）的详细介绍，探讨了人工智能在医疗、金融、教育、交通等领域的应用，并分析了人工智能发展过程中面临的挑战与机遇。最后，本文对人工智能的未来发展进行了展望，提出了可能的突破方向。 1. 引言
人工智能是指通过计算机程序模拟人类智能的一门科学。自20世纪50年代诞生以来，人工智能经历了多次起伏，近年来随着计算能力的提升和大数据的普及，人工智能技术取得了显著的进展。人工智能的应用已经渗透到日常生活的方方面面，从智能手机的语音助手到自动驾驶汽车，从医疗诊断到金融分析，人工智能正在改变着人类社会的运行方式。'
🔍 文書: page_content='2. 人工智能的发展历程
2.1 早期发展
人工智能的概念最早可以追溯到20世纪50年代。1956年，达特茅斯会议（Dartmouth Conference）被认为是人工智能研究的正式开端。在随后的几十年里，人工智能研究经历了多次高潮与低谷。早期的研究主要集中在符号逻辑和专家系统上，但由于计算能力的限制和算法的不足，进展缓慢。 2.2 机器学习的兴起
20世纪90年代，随着统计学习方法的引入，机器学习逐渐成为人工智能研究的主流。支持向量机（SVM）、决策树、随机森林等算法在分类和回归任务中取得了良好的效果。这一时期，机器学习开始应用于数据挖掘、模式识别等领域。'
🔍 文書: page_content='2.3 深度学习的突破
2012年，深度学习在图像识别领域取得了突破性进展，标志着人工智能进入了一个新的阶段。深度学习通过多层神经网络模拟人脑的工作方式，能够自动提取特征并进行复杂的模式识别。卷积神经网络（CNN）、循环神经网络（RNN）和长短期记忆网络（LSTM）等深度学习模型在图像处理、自然语言处理、语音识别等领域取得了显著成果。 3. 人工智能的核心技术
3.1 机器学习
机器学习是人工智能的核心技术之一，通过算法使计算机从数据中学习并做出决策。常见的机器学习算法包括监督学习、无监

## 5、HTMLHeaderTextSplitter（参考程度）

例

In [11]:
# 1.関連する依存関係をインポート
from langchain_text_splitters import HTMLHeaderTextSplitter

# 2.HTML ファイルを定義
html_string = """
<!DOCTYPE html>
<html>
<body>
    <div>
        <h1>ようこそ尚硅谷へ！</h1>
        <p>尚硅谷はIT技術分野の専門トレーニング機関です</p>
        <div>
            <h2>尚硅谷講師紹介</h2>
            <p>尚硅谷の講師は長年の指導経験を持ち、皆現場の第一線のIT企業出身です</p>
            <h3>尚硅谷北京キャンパス</h3>
            <p>北京キャンパスは宏福科技園区にあります</p>
        </div>
    </div>
</body>
</html>
"""

# 4.どの HTML タグに基づいてテキストを分割するかを指定
headers_to_split_on = [
    ("h1", "見出し1"),
    ("h2", "見出し2"),
    ("h3", "見出し3"),
]

# 5.HTMLHeaderTextSplitter 分割器を定義
html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# 6.分割器で分割
html_header_splits = html_splitter.split_text(html_string)

print(html_header_splits)

[Document(metadata={'見出し1': 'ようこそ尚硅谷へ！'}, page_content='ようこそ尚硅谷へ！'), Document(metadata={'見出し1': 'ようこそ尚硅谷へ！'}, page_content='尚硅谷はIT技術分野の専門トレーニング機関です'), Document(metadata={'見出し1': 'ようこそ尚硅谷へ！', '見出し2': '尚硅谷講師紹介'}, page_content='尚硅谷講師紹介'), Document(metadata={'見出し1': 'ようこそ尚硅谷へ！', '見出し2': '尚硅谷講師紹介'}, page_content='尚硅谷の講師は長年の指導経験を持ち、皆現場の第一線のIT企業出身です'), Document(metadata={'見出し1': 'ようこそ尚硅谷へ！', '見出し2': '尚硅谷講師紹介', '見出し3': '尚硅谷北京キャンパス'}, page_content='尚硅谷北京キャンパス'), Document(metadata={'見出し1': 'ようこそ尚硅谷へ！', '見出し2': '尚硅谷講師紹介', '見出し3': '尚硅谷北京キャンパス'}, page_content='北京キャンパスは宏福科技園区にあります')]


## 6、CodeTextSplitter（参考程度）

例1：サポートされている言語

In [12]:
from langchain_text_splitters import Language

# サポートされている分割言語タイプ
# Full list of supported languages
langs = [e.value for e in Language]
print(langs)

['cpp', 'go', 'java', 'kotlin', 'js', 'ts', 'php', 'proto', 'python', 'r', 'rst', 'ruby', 'rust', 'scala', 'swift', 'markdown', 'latex', 'html', 'sol', 'csharp', 'cobol', 'c', 'lua', 'perl', 'haskell', 'elixir', 'powershell', 'visualbasic6']


例2：

In [13]:
# 1.関連する依存関係をインポート
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from pprint import pprint

# 2.分割する Python コード片を定義
PYTHON_CODE = """
def hello_world():
    print("Hello, World!")

def hello_world1():
    print("Hello, World1!")
"""

# 3.再帰的文字分割器を定義
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=50,
    chunk_overlap=0
)

# 4.文書分割
python_docs = python_splitter.create_documents(texts=[PYTHON_CODE])

pprint(python_docs)

[Document(metadata={}, page_content='def hello_world():\n    print("Hello, World!")'),
 Document(metadata={}, page_content='def hello_world1():\n    print("Hello, World1!")')]


## 7、MarkdownTextSplitter（参考程度）

例：

In [14]:
from langchain_text_splitters import MarkdownTextSplitter

markdown_text = """
# レベル1見出し\n
これはレベル1見出し下の内容です\n\n
## レベル2見出し\n
- レベル2のリスト項目1\n
- レベル2のリスト項目2\n
"""

# 重要なステップ：インスタンス属性を直接変更
splitter = MarkdownTextSplitter(chunk_size=30, chunk_overlap=0)
splitter._is_separator_regex = True  #  区切り文字を強制的に正規表現として扱う

# 分割を実行
docs = splitter.create_documents(texts = [markdown_text])

# print(len(docs))

for i, doc in enumerate(docs):
    print(f"\n🔍 チャンク {i + 1}:")
    print(doc.page_content)


🔍 チャンク 1:
# レベル1見出し

これはレベル1見出し下の内容です

🔍 チャンク 2:
## レベル2見出し

- レベル2のリスト項目1

🔍 チャンク 3:
- レベル2のリスト項目2
